In [0]:
import time

import pyspark.sql.functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG = 'car_workshop'
LAB = f'{CATALOG}.lab'

tables = spark.sql(f'SHOW TABLES IN {LAB}').collect()

for table_row in tables:
    spark.sql(f'DROP TABLE IF EXISTS {LAB}.{table_row.tableName}')
    print(f"dropped {table_row}")


spark.sql(f'DROP VOLUME IF EXISTS {LAB}.files')
spark.sql(f'DROP SCHEMA IF EXISTS {LAB}')
print(f"{LAB} is now clear, ready for recreation")

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {LAB}')
spark.sql(f'CREATE VOLUME IF NOT EXISTS {LAB}.files')
LAB_DIR = f'/Volumes/{CATALOG}/lab/files'

print(f"{LAB} is created")


def timed(label, fn):
    t0 = time.time()
    result = fn()
    print(f'{label}: {time.time() - t0:.1f}s')
    return result


print(f'lab schema: {LAB}, lab volume: {LAB_DIR}')

dbutils.widgets.dropdown("Is cluster mode?", "False", ["True","False"])
is_cluster_mode = dbutils.widgets.get("Is cluster mode?")

In [0]:
fact_schema = f"{CATALOG}.fact"
tables = spark.sql(f'SHOW TABLES IN {fact_schema}').collect()

for table_row in tables:
    table = spark.sql(f'DESCRIBE TABLE {fact_schema}.{table_row.tableName}')
    print(f"table: {table_row}")
    display(table)

In [0]:
%sql
select distinct sale_date from car_workshop.fact.fact_invoices 
order by sale_date asc

In [0]:
%sql
with cte as (
select 
*,
month(sale_date) as sales_month,
case when month(sale_date) between 1 and 8  then 'skew_1'
when month(sale_date) between 8 and 9 then 'skew_2'
when month(sale_date) between 10 and 11 then 'skew_3'
when month(sale_date) between 12 and 12 then 'skew_4'
end as skew_test
 from car_workshop.fact.fact_invoices)
 , union_1 as (select * from cte where skew_test = 'skew_1')
 , union_2 as (
 select skew_test, count(skew_test) as skew_count from cte
  group by skew_test
 union all 
 select skew_test, count(skew_test) as skew_count from union_1
 group by skew_test
  union all 
 select skew_test, count(skew_test) as skew_count from union_1
 group by skew_test
   union all 
 select skew_test, count(skew_test) as skew_count from union_1
 group by skew_test
   union all 
 select skew_test, count(skew_test) as skew_count from union_1
 group by skew_test
   union all 
 select skew_test, count(skew_test) as skew_count from union_1
 group by skew_test
   union all 
 select skew_test, count(skew_test) as skew_count from union_1
 group by skew_test
   union all 
 select skew_test, count(skew_test) as skew_count from union_1
 group by skew_test
   union all 
 select skew_test, count(skew_test) as skew_count from union_1
 group by skew_test)
 select skew_test, sum(skew_count) as skew_sum from union_2
 group by skew_test


In [0]:
%sql
create table car_workshop.lab.fact_invoices as
with cte as (
  select 
    *,
    month(sale_date) as sales_month,
    case when month(sale_date) between 1 and 8  then 'skew_1'
    when month(sale_date) between 8 and 9 then 'skew_2'
    when month(sale_date) between 10 and 11 then 'skew_3'
    when month(sale_date) between 12 and 12 then 'skew_4'
    end as skew_test
  from car_workshop.fact.fact_invoices
),
skew_1_multiplied as (
  select cte.* 
  from cte
  CROSS JOIN LATERAL explode(sequence(1, 25)) AS t(multiplier)
  where skew_test = 'skew_1'
),
other_skews as (
  select *
  from cte
  where skew_test != 'skew_1'
),
combined as (
  select * from skew_1_multiplied
  union all
  select * from other_skews
)
-- check the skew
-- select skew_test, count(skew_test) as row_count 
-- from combined
-- group by skew_test
select * from combined

In [0]:
%sql
select skew_test, count(*) from car_workshop.lab.fact_invoices
group by skew_test

In [0]:
%sql
select count(*) from car_workshop.lab.fact_invoices